# 01b · Extracción extendida desde 1985 (datos crudos)
**Tesis:** Medición del ciclo financiero en Perú mediante técnicas de reducción dimensional y machine learning
**Autor:** Roberto Samaniego Salcedo · **Asesor:** Dr. Sergio Camiz

---

**Objetivo:** ampliar la ventana histórica hacia atrás tanto como lo permita cada serie del
BCRP, a partir de **enero de 1985**, sin modificar el pipeline vigente.

**Por qué:**

1. Autoencoders y LSTM necesitan más observaciones: con 2007–2025 solo hay ~200 meses, y la
   validación walk-forward de `09` no puede evaluar la crisis de 2008–2009 fuera de muestra.
2. Hipótesis de trabajo del proyecto: la política económica peruana debe leerse desde la
   transición de los años ochenta. La duración del ciclo de Drehmann, Borio & Tsatsaronis
   (2012), de 8 a 30 años, exige ventanas largas.

**Principio de responsabilidad única (igual que `01`):** este notebook **solo descarga y
diagnostica**. No resamplea, no imputa y no transforma. Las salidas son archivos nuevos;
`data/raw/series_bcrp_raw.csv` no se toca.

**Salidas:**

- `data/raw/series_bcrp_raw_extendido.csv` (mensual, desde 1985)
- `data/raw/embi_peru_raw_extendido.csv` (diario, desde su primer dato disponible)
- `data/results/01b_cobertura_extendida.csv` (diagnóstico por serie)

::: {.callout-warning title="Riesgos conocidos al ampliar la muestra"}
- **Cambios de régimen monetario.** Hiperinflación (1988–1990), estabilización y reformas
  (1990–2001) y metas explícitas de inflación (desde 2002). Mezclarlos sin control puede
  hacer que el PCA o el autoencoder solo "descubran" la hiperinflación. Esto se trata en `04c`.
- **Cambio de unidad monetaria.** Inti (1985–1991) y nuevo sol (desde julio de 1991). El BCRP
  suele publicar re-expresado en soles, con valores muy pequeños antes de 1991. Se diagnostica
  la **resolución numérica**: ceros y valores repetidos por redondeo.
- **Series que no existen hacia atrás.** El crédito por tipo, la tasa de referencia y el EMBI
  tienen inicios tardíos. El diagnóstico de cobertura lo muestra serie por serie; no se rellena nada.
:::

> **Ejecución:** este notebook requiere acceso a la API del BCRP; se ejecuta en la PC local.

## 1. Librerías y raíz del repositorio

In [ ]:
import os, time
import requests
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

def ir_a_raiz_del_repo(marcador='_quarto.yml'):
    """Sube directorios hasta encontrar el marcador; es idempotente."""
    actual = os.path.abspath(os.getcwd())
    while not os.path.exists(os.path.join(actual, marcador)):
        padre = os.path.dirname(actual)
        if padre == actual:
            raise FileNotFoundError(marcador)
        actual = padre
    os.chdir(actual)

ir_a_raiz_del_repo()
print('Directorio:', os.getcwd())

## 2. Series a descargar

Se usan **exactamente los mismos códigos** que `01_extraccion_bcrp` (verificados contra
BCRPData), ahora desde 1985. Si alguna serie no tiene historia larga, la API devuelve sus
datos desde su primer periodo disponible y el diagnóstico de cobertura lo reporta.

El diccionario `SERIES_CANDIDATAS_HISTORIA_LARGA` queda **vacío a propósito**. Allí se
agregan, *después de verificarlos en el buscador de BCRPData*, códigos alternativos con más
historia para las variables que empiecen tarde. Por ejemplo, el crédito al sector privado en
moneda nacional y extranjera como sustituto del desglose por tipo. No se incluye ningún
código sin verificar.

In [ ]:
SERIES = {
    # CRÉDITO (Cuadro 19 BCRP)
    'Crédito SF sector privado': 'PN00518MM',
    'Crédito empresarial':       'PN00532MM',
    'Crédito consumo':           'PN00533MM',
    'Crédito hipotecario':       'PN00534MM',
    # TASAS
    'Tasa activa TAMN':          'PN07807NM',
    'Tasa pasiva TIPMN':         'PN07816NM',
    'Tasa referencia BCRP':      'PD04722MM',
    'Tasa interbancaria':        'PN07819NM',
    # TIPO DE CAMBIO, MERCADO Y PRECIOS
    'Tipo de cambio':            'PN01246PM',
    'Índice BVL':                'PN01142MM',
    'IPC Lima':                  'PN38705PM',
    'IPC Subyacente':            'PN38708PM',
    # ACTIVIDAD
    'PBI desestacionalizado':    'PN01773AM',
    'Demanda interna':           'PN01774AM',
    'Reservas internacionales':  'PN00027MM',
    # LIQUIDEZ
    'Liquidez M1':               'PN00199MM',
    'Liquidez M2':               'PN00208MM',
    'Liquidez M3':               'PN00214MM',
}

# Completar SOLO con códigos verificados en https://estadisticas.bcrp.gob.pe/estadisticas/series/buscador
# Formato: 'Nombre descriptivo (fuente larga)': 'CODIGO'
SERIES_CANDIDATAS_HISTORIA_LARGA = {
}

INICIO = '1985-1'
FIN = pd.Timestamp.today().strftime('%Y-') + str(pd.Timestamp.today().month)
print(f'Periodo solicitado: {INICIO} a {FIN} | {len(SERIES) + len(SERIES_CANDIDATAS_HISTORIA_LARGA)} series')

## 3. Funciones de descarga y parseo robusto

Corrige el problema detectado el 2026-10-04: el BCRP rotula septiembre como **"Set"**, que
`pd.to_datetime` no reconoce. Con `errors='coerce'`, esa etiqueta se convertía en `NaT` sin
aviso, y así se perdieron los septiembres de la serie diaria del EMBI. Aquí cualquier
etiqueta no parseada **detiene la ejecución** en lugar de borrarse en silencio.

In [ ]:
BASE_URL = 'https://estadisticas.bcrp.gob.pe/estadisticas/series/api'
EQUIVALENCIAS_MES = {'Ene': 'Jan', 'Abr': 'Apr', 'Ago': 'Aug', 'Sept': 'Sep', 'Set': 'Sep', 'Dic': 'Dec'}

def parsear_periodos_bcrp(etiquetas, codigo=''):
    """
    Convierte etiquetas de periodo del BCRP ('Set.2007', '02Set07', 'Jan.1985', ...) a fechas.
    Normaliza las abreviaturas en español en cualquier posición y falla si queda algún NaT.
    """
    etiquetas = pd.Series(etiquetas, dtype=str)
    for es, en in EQUIVALENCIAS_MES.items():
        etiquetas = etiquetas.str.replace(rf'(?<![A-Za-z]){es}(?![a-z])', en, regex=True)
    fechas = pd.to_datetime(etiquetas, format='mixed', errors='coerce')
    if fechas.isna().any():
        raise ValueError(f'{codigo}: {fechas.isna().sum()} periodos sin parsear, '
                         f'p. ej. {etiquetas[fechas.isna()].head(3).tolist()}')
    return fechas

def descargar_serie(codigo, inicio, fin, reintentos=3):
    """
    Descarga una serie del BCRP y la devuelve como Series indexada por fecha.
    Los valores 'n.d.' se mantienen como NaN (no se imputan).
    """
    url = f'{BASE_URL}/{codigo}/json/{inicio}/{fin}/ing'
    for intento in range(reintentos):
        try:
            r = requests.get(url, timeout=30)
            r.raise_for_status()
            break
        except requests.RequestException:
            if intento == reintentos - 1:
                raise
            time.sleep(2 * (intento + 1))
    periodos = r.json().get('periods', [])
    if not periodos:
        raise ValueError(f'La API del BCRP devolvió 0 periodos para {codigo}')
    etiquetas = [p['name'] for p in periodos]
    valores = []
    for p in periodos:
        try:
            valores.append(float(str(p['values'][0]).replace(',', '.')))
        except (ValueError, TypeError):
            valores.append(np.nan)            # 'n.d.' u otro texto -> NaN explícito
    return pd.Series(valores, index=parsear_periodos_bcrp(etiquetas, codigo), name=codigo)

## 4. Descarga

In [ ]:
descargadas, errores = {}, []
for nombre, codigo in {**SERIES, **SERIES_CANDIDATAS_HISTORIA_LARGA}.items():
    try:
        s = descargar_serie(codigo, INICIO, FIN)
        descargadas[nombre] = s
        primero = s.first_valid_index()
        print(f'  OK     {nombre:28s} ({codigo}): {s.notna().sum():4d} obs. desde {primero:%Y-%m}')
    except Exception as e:
        errores.append((nombre, codigo, str(e)))
        print(f'  ERROR  {nombre:28s} ({codigo}): {e}')

df_raw_ext = pd.DataFrame(descargadas).sort_index()
# Las series mensuales del BCRP vienen fechadas al día 1; se fuerza inicio de mes por seguridad
df_raw_ext.index = df_raw_ext.index.to_period('M').to_timestamp()
print(f'\nConsolidado: {df_raw_ext.shape[0]} meses x {df_raw_ext.shape[1]} series '
      f'({df_raw_ext.index.min():%Y-%m} a {df_raw_ext.index.max():%Y-%m})')
if errores:
    print(f'Series con error: {len(errores)} — revisar antes de continuar')

### 4.1 EMBI Perú (diario, desde su primer dato)

El índice EMBIG empieza a fines de los noventa; se solicita desde 1985 y la API devuelve lo
disponible. Para la versión extendida del panel, `04c` decidirá si se excluye el EMBI o si se
recorta la muestra.

In [ ]:
os.makedirs('data/raw', exist_ok=True)
try:
    embi = descargar_serie('PD04709XD', '1985-1-1', pd.Timestamp.today().strftime('%Y-%m-%d'))
    embi.to_frame('embi_peru').to_csv('data/raw/embi_peru_raw_extendido.csv')
    meses_embi = embi.resample('MS').count()
    print(f'EMBI: {embi.notna().sum()} días, desde {embi.first_valid_index():%Y-%m-%d}')
    print(f'Meses con al menos un dato: {(meses_embi > 0).sum()} — meses sin datos dentro del rango: '
          f'{(meses_embi.loc[embi.first_valid_index():] == 0).sum()}')
except Exception as e:
    print('ERROR EMBI:', e)

## 5. Diagnóstico de cobertura y resolución numérica

Para cada serie se reporta:

- **inicio / fin:** primer y último dato no nulo;
- **huecos internos:** meses NaN entre el inicio y el fin. Un hueco interno rompe la
  continuidad de las diferencias;
- **ceros y valores repetidos antes de 1992:** señal de pérdida de resolución por la
  re-expresión de intis a soles. Con pocos decimales, una serie en millones de soles de 1985
  puede quedar redondeada a 0,0001 o a 0, y su logaritmo pierde sentido.

In [ ]:
def diagnostico(s):
    v = s.loc[s.first_valid_index():s.last_valid_index()] if s.notna().any() else s
    pre92 = v.loc[:'1991-12']
    return pd.Series({
        'inicio': v.first_valid_index(),
        'fin': v.last_valid_index(),
        'meses_con_dato': int(v.notna().sum()),
        'huecos_internos': int(v.isna().sum()),
        'ceros': int((v == 0).sum()),
        'no_positivos': int((v <= 0).sum()),
        'valores_distintos_pre1992': int(pre92.nunique()),
        'meses_pre1992': int(pre92.notna().sum()),
        'minimo': v.min(), 'maximo': v.max(),
    })

cobertura = df_raw_ext.apply(diagnostico).T.sort_values('inicio')
cobertura['anios'] = (cobertura['meses_con_dato'] / 12).astype(float).round(1)
os.makedirs('data/results', exist_ok=True)
cobertura.to_csv('data/results/01b_cobertura_extendida.csv')
cobertura[['inicio', 'fin', 'meses_con_dato', 'anios', 'huecos_internos', 'ceros',
           'no_positivos', 'valores_distintos_pre1992', 'meses_pre1992']]

In [ ]:
# Mapa de disponibilidad: proporción de meses con dato por año y serie
disp = df_raw_ext.notna().groupby(df_raw_ext.index.year).mean().T.loc[cobertura.index]
fig, ax = plt.subplots(figsize=(12, 0.35 * len(disp) + 1.5))
im = ax.imshow(disp.values, aspect='auto', cmap='Greens', vmin=0, vmax=1)
ax.set_yticks(range(len(disp))); ax.set_yticklabels(disp.index, fontsize=8)
ax.set_xticks(range(0, disp.shape[1], 2)); ax.set_xticklabels(disp.columns[::2], rotation=90, fontsize=8)
for x, etiqueta in [(1990, 'Ago-1990'), (1991, 'Nuevo sol'), (2002, 'Metas de inflación')]:
    if x in disp.columns:
        i = list(disp.columns).index(x)
        ax.axvline(i - 0.5, color='crimson', lw=1, ls='--')
        ax.text(i - 0.4, -0.7, etiqueta, color='crimson', fontsize=7)
plt.colorbar(im, ax=ax, label='Proporción de meses con dato')
ax.set_title('Disponibilidad de series BCRP desde 1985')
plt.tight_layout(); os.makedirs('reports/figures', exist_ok=True)
plt.savefig('reports/figures/01b_disponibilidad.png'); plt.show()

## 6. Guardar datos crudos extendidos (inmutables)

Archivo nuevo; el `series_bcrp_raw.csv` vigente no se sobrescribe. Los `.csv` de
`data/raw/` están en `.gitignore`: la reproducibilidad se garantiza con este notebook y con
el diagnóstico de cobertura, que sí se versiona.

In [ ]:
os.makedirs('data/raw', exist_ok=True)
df_raw_ext.to_csv('data/raw/series_bcrp_raw_extendido.csv')
print(f'Guardado: data/raw/series_bcrp_raw_extendido.csv ({df_raw_ext.shape[0]} x {df_raw_ext.shape[1]})')
print('Siguiente paso: 04c_preprocesamiento_extendido.ipynb')